In [42]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split,cross_val_score,GridSearchCV
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.metrics import classification_report,confusion_matrix,roc_auc_score

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score 
from sklearn.ensemble import RandomForestClassifier





In [3]:
#read dataframe

df=pd.read_csv("car_cleaned.csv")

In [4]:
df.head()

,Unnamed: 0,age,gender,driving_experience,education,income,credit_score,vehicle_ownership,vehicle_year,married,children,postal_code,annual_mileage,vehicle_type,speeding_violations,duis,past_accidents,outcome
0,0,3,0,0,1,3,0.629027,1.0,1,0.0,1.0,10238,12000.0,0,0,0,0,0.0
1,1,0,1,0,0,0,0.357757,0.0,0,0.0,0.0,10238,16000.0,0,0,0,0,1.0
2,2,0,0,0,1,1,0.493146,1.0,0,0.0,0.0,10238,11000.0,0,0,0,0,0.0
3,3,0,1,0,2,1,0.206013,1.0,0,0.0,1.0,32765,11000.0,0,0,0,0,0.0
4,4,1,1,1,0,1,0.388366,1.0,0,0.0,0.0,32765,12000.0,0,2,0,1,1.0


In [5]:
#dropping unwantd columns

df.drop(columns=["Unnamed: 0","postal_code"],inplace=True)

In [6]:
#numeric columns that must be standerdised
num_col=["speeding_violations","duis","past_accidents","annual_mileage"]

In [7]:
#creating a pipe line
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_col)       
       
    ],
    remainder="passthrough"  # keep other columns (like violations, accidents) as they are
)



In [8]:
#feature
X=df.drop(columns="outcome")

#target
y=df["outcome"]


In [9]:
#crating test and training dataset

x_train,x_test,y_train,y_test=train_test_split(X,y,random_state=42,test_size=0.2)

**CREATING BASELINE MODEL**

In [30]:
# Full pipeline: preprocessing + model
LR = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000,))
])

In [25]:
#TRAINING PIPELINE

LR.fit(x_train,y_train);

**EVALUATING THE MODEL**

In [26]:
#EVALUATING THE MODEL
y_predi=LR.predict(x_test)
#classification report
print(classification_report(y_test,y_predi))

              precision    recall  f1-score   support

         0.0       0.87      0.91      0.89      1367
         1.0       0.78      0.70      0.74       633

    accuracy                           0.84      2000
   macro avg       0.83      0.81      0.81      2000
weighted avg       0.84      0.84      0.84      2000



The model achieved an overall accuracy of 84% across 2,000 records. For the majority class (non‑claims), performance was strong with precision of 0.87, recall of 0.91, and an F1‑score of 0.89, indicating the model is highly effective at correctly identifying non‑claims. For the minority class (claims), performance was weaker: precision of 0.78, recall of 0.70, and an F1‑score of 0.74, meaning the model misses about 30% of actual claims. The macro average F1‑score of 0.81 highlights the performance gap between classes, while the weighted average of 0.84 reflects the influence of the larger non‑claim class. Overall, the model is reliable but shows room for improvement in capturing claims, which are the more critical outcomes from a business perspective.


In [27]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Model


# Cross-validation with stratification
scores = cross_val_score(LR, X, y, cv=skf, scoring='roc_auc')

print("Accuracy scores across folds:", scores)
print("Mean accuracy:", scores.mean())
print("Standard deviation:", scores.std())



Accuracy scores across folds: [0.9031814  0.90192054 0.89780931 0.89516722 0.89725522]
Mean accuracy: 0.8990667384888736
Standard deviation: 0.003004782120511018


The model’s accuracy across the five folds was consistently high, ranging from 89.5% to 90.3%. The mean accuracy was approximately 89.9%, with a very low standard deviation of 0.003, indicating that performance is stable and does not vary much across different train/test splits. This consistency suggests the model generalizes well and is not overly sensitive to how the data is partitioned


**TUNNING MY BASELINE MODEL**

In [32]:
# Hyperparameter grid
param_grid = {
    'classifier__C': [0.01, 0.1, 1, 10, 100],
    'classifier__penalty': ['l1', 'l2'],
    'classifier__solver': ['liblinear', 'saga'],
    'classifier__class_weight': [None, 'balanced']
}


# Grid Search with cross-validation
grid_search = GridSearchCV(
    estimator=LR,
    param_grid=param_grid,
    cv=skf,
    scoring='recall',        # optimize for F1-score (good for imbalanced claims)
    n_jobs=-1
)

# Fit
grid_search.fit(X, y)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validated recall score:", grid_search.best_score_)


Best parameters: {'classifier__C': 0.1, 'classifier__class_weight': 'balanced', 'classifier__penalty': 'l1', 'classifier__solver': 'saga'}
Best cross-validated F1 score: 0.8458316135968733


We selected recall as the primary scoring metric during hyperparameter tuning because in the claim prediction context, missing a true claim (false negative) is more costly than incorrectly flagging a non-claim (false positive).

**TRAINING BASELINE WITH NEW PARAMETER**

In [34]:
LR_TUNNED = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000,C=0.1,class_weight="balanced",penalty='l1',solver='saga'))
])

In [35]:
LR_TUNNED.fit(x_train,y_train);

In [36]:
#EVALUATING THE MODEL
y_predi=LR_TUNNED.predict(x_test)
#classification report
print(classification_report(y_test,y_predi))

              precision    recall  f1-score   support

         0.0       0.91      0.81      0.86      1367
         1.0       0.67      0.83      0.74       633

    accuracy                           0.81      2000
   macro avg       0.79      0.82      0.80      2000
weighted avg       0.83      0.81      0.82      2000



After hyperparameter tuning, the model achieved an overall accuracy of 81% across 2,000 records. For the majority class (non‑claims), precision improved to 0.91, meaning predictions of non‑claims are highly reliable, though recall dropped to 0.81, indicating some non‑claims were missed. For the minority class (claims), recall increased to 0.83, showing the model is now much better at capturing actual claims, though precision decreased to 0.67, meaning more non‑claims are incorrectly flagged as claims. The F1‑score for claims rose to 0.74, reflecting a stronger balance between precision and recall compared to before tuning. The macro averages (≈0.80) highlight improved balance across both classes, while the weighted averages (≈0.82) reflect the influence of the larger non‑claim class.


In [41]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Model


# Cross-validation with stratification
scores = cross_val_score(LR_TUNNED, X, y, cv=skf, scoring='roc_auc')

print("Accuracy scores across folds:", scores)
print("Mean accuracy:", scores.mean())
print("Standard deviation:", scores.std())



Accuracy scores across folds: [0.90290179 0.90222282 0.89820368 0.8945748  0.89732782]
Mean accuracy: 0.8990461818515494
Standard deviation: 0.0031180954550406397


**TRAINING  RANDOM FOREST MODEL**

In [56]:
RF= Pipeline([
    ("preprocessing", preprocessor),
    ("classifier",RandomForestClassifier())
])

In [48]:
RF.fit(x_train,y_train)

C:\Users\precious\anaconda3\Lib\site-packages\sklearn\compose\_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['speeding_violations',
                                                   'duis', 'past_accidents',
                                                   'annual_mileage'])])),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced', max_depth=20,
                                        n_estimators=200, random_state=42))])

In [49]:
#EVALUATING THE MODEL
y_predi=RF.predict(x_test)
#classification report
print(classification_report(y_test,y_predi))

              precision    recall  f1-score   support

         0.0       0.84      0.90      0.87      1367
         1.0       0.75      0.64      0.69       633

    accuracy                           0.82      2000
   macro avg       0.80      0.77      0.78      2000
weighted avg       0.82      0.82      0.82      2000



In [51]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Model


# Cross-validation with stratification
scores = cross_val_score(RF, X, y, cv=skf, scoring='roc_auc')

print("Accuracy scores across folds:", scores)
print("Mean accuracy:", scores.mean())
print("Standard deviation:", scores.std())



Accuracy scores across folds: [0.885184   0.888251   0.87761988 0.87713374 0.87552316]
Mean accuracy: 0.8807423541831103
Standard deviation: 0.005022346668023499


In [ ]:
# Hyperparameter grid

param_grid = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [None, 10, 20, 30],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4],
    'classifier__max_features': ['sqrt', 'log2', None],
    'classifier__bootstrap': [True, False],
    'classifier__class_weight': [None, 'balanced']
}


# Grid Search with cross-validation
grid_search = GridSearchCV(
    estimator=RF,
    param_grid=param_grid,
    cv=skf,
    scoring='recall',        # optimize for F1-score (good for imbalanced claims)
    n_jobs=-1
)

# Fit
grid_search.fit(X, y)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validated recall score:", grid_search.best_score_)
